In [ ]:
import marimo as mo

# One-loop QCD: gluon self-energy numerators

Generate the three non-scaleless one-loop QCD contributions to the gluon
two-point function: a gluon loop, a Faddeev–Popov ghost loop, and a
bottom-quark loop. Then choose a graph interactively and inspect its
instantiated Feynman rules as native Symbolica expressions.

A full Standard-Model generation would produce one quark loop per flavor.
Here we retain only `b`/`b~` at generation time, so the three displayed
graphs correspond directly to the three familiar loop-field classes.

In [ ]:
from pathlib import Path

import symbolica.community.feynkit as fk

data_dir = next(
    path
    for path in (Path("data"), Path("examples/feynkit/data"))
    if (path / "sm.json").is_file()
)

## Load FeynKit's normalized Standard Model

The bundled `sm.json` is copied byte-for-byte from FeynKit's authoritative
normalized model fixture. Loading JSON keeps this example deterministic and
avoids the optional Python UFO loader. We use only its QCD sector below.

In [ ]:
model = fk.Model(data_dir / "sm.json")
gluon = model.particle("g")

mo.ui.table(
    [
        {
            "model": model.name,
            "particles": len(model.particles),
            "interaction rules": len(model.vertex_rules),
            "gluon PDG": gluon.pdg_code,
            "gluon antiparticle": gluon.antiparticle.name,
            "gluon massless": gluon.is_massless,
        }
    ]
)

## Generate the one-loop QCD gluon self-energy

`loops=1` fixes the loop order, while the coupling-order bounds select
exactly $g_s^2$ and exclude electroweak insertions. `add_particle_veto`
removes the other five quark flavors before graph generation. We look up
each quark as a concrete `Particle` and obtain its partner through
`particle.antiparticle`; the generator likewise accepts the concrete gluon
objects as its external states. Self-loops stay disabled, excluding the
massless four-gluon tadpole, which is scaleless and vanishes in dimensional
regularization.

FeynKit automatically instantiates each vertex and propagator rule while
building the diagrams.

In [ ]:
options = fk.GenerationOptions(max_vertices=2)
options.set_coupling_orders({"QCD": (2, 2), "QED": (0, 0)})
quarks_to_veto = [
    model.particle(name) for name in ("d", "u", "s", "c", "t")
]
options.add_particle_veto(
    [
        particle
        for quark in quarks_to_veto
        for particle in (quark, quark.antiparticle)
    ]
)

generated = model.generate_diagrams(
    incoming=[gluon],
    outgoing=[gluon.antiparticle],
    loops=1,
    options=options,
)

mo.ui.table(
    [
        {
            "completed": generated.report.completed,
            "retained diagrams": len(generated),
            "topologies considered": generated.report.topology_count,
            "interaction assignments": (
                generated.report.interaction_assignment_count
            ),
        }
    ]
)

In [ ]:
def internal_edges(
    diagram: fk.FeynmanDiagram,
) -> list[fk.DiagramEdge]:
    external_vertices = {
        vertex.id for vertex in diagram.vertices if vertex.is_external
    }
    return [
        edge
        for edge in diagram.edges
        if edge.source not in external_vertices
        and edge.target not in external_vertices
    ]

## The three loop-field contributions

Internal edge metadata identifies the loop field without parsing graph
labels. The rows are ordered as gluon, ghost, and bottom-quark contributions
even though generator names are assigned independently of presentation.
Each `diagram` cell delegates to `FeynmanDiagram._repr_html_()`, producing a
Linnest-rendered graph rather than a string snapshot.

In [ ]:
graph_catalog = (
    ("gluon", "Gluon loop", 21, "g"),
    ("ghost", "Ghost loop", 9000005, "ghG / ghG~"),
    ("bottom", "Bottom-quark loop", 5, "b / b~"),
)
_diagram_by_pdg = {
    abs(internal_edges(_diagram)[0].particle_pdg): _diagram
    for _diagram in generated.diagrams
    if internal_edges(_diagram)
}
_expected_pdgs = {_pdg for _, _, _pdg, _ in graph_catalog}
if len(generated) != 3 or set(_diagram_by_pdg) != _expected_pdgs:
    raise RuntimeError(
        "expected exactly the gluon, ghost, and bottom-quark bubbles"
    )

diagrams_by_kind = {
    _kind: _diagram_by_pdg[_pdg]
    for _kind, _, _pdg, _ in graph_catalog
}
labels_by_kind = {
    _kind: _label for _kind, _label, _, _ in graph_catalog
}

mo.ui.table(
    [
        {
            "contribution": _label,
            "loop field": _field,
            "diagram": mo.as_html(diagrams_by_kind[_kind]),
        }
        for _kind, _label, _, _field in graph_catalog
    ],
    column_widths={"contribution": 190, "loop field": 130, "diagram": 520},
)

## Choose a graph

`graph_index` defaults to the first row, the gluon loop. Set it to `1` for
the ghost loop or `2` for the bottom-quark loop, then rerun this and the
following cells.

In [ ]:
graph_index = 0  # 0: gluon, 1: ghost, 2: bottom quark
if graph_index not in range(len(graph_catalog)):
    raise IndexError("graph_index must be 0, 1, or 2")

selected_kind = graph_catalog[graph_index][0]
selected_diagram = diagrams_by_kind[selected_kind]
selected_label = labels_by_kind[selected_kind]
{"graph_index": graph_index, "selected contribution": selected_label}

In [ ]:
mo.vstack(
    [
        mo.md(f"### {selected_label}"),
        mo.as_html(selected_diagram),
    ],
    align="center",
    gap=1,
)

## Instantiated numerator

The combined numerator is the native Symbolica product of the selected
graph's interaction and internal-propagator factors. The diagram-wide
factor contains its automorphism factor, external-fermion ordering sign,
and the minus sign for each closed internal fermion loop.
Multiplying the two gives the numerator with this combinatorial factor
included.

In [ ]:
selected_numerator = selected_diagram.numerator_expression()
selected_factor = selected_diagram.overall_factor_expression()
weighted_numerator = selected_factor * selected_numerator

mo.ui.table(
    [
        {
            "contribution": selected_label,
            "diagram factor": mo.as_html(selected_factor),
            "analytic numerator": mo.as_html(selected_numerator),
            "with diagram factor": mo.as_html(weighted_numerator),
        }
    ],
    column_widths={
        "contribution": 180,
        "diagram factor": 230,
        "analytic numerator": 650,
        "with diagram factor": 650,
    },
)

## Feynman-rule factors

The combined expression can also be inspected rule by rule. Only internal
vertices and propagators appear here; every factor remains a native
Symbolica expression with rich mathematical rendering.

In [ ]:
_vertex_rows = [
    {
        "vertex": vertex.id,
        "interaction rule": vertex.interaction,
        "analytic factor": mo.as_html(vertex.numerator_expression()),
    }
    for vertex in selected_diagram.vertices
    if not vertex.is_external
]
_propagator_rows = [
    {
        "edge": edge.id,
        "loop field": edge.particle_name,
        "analytic factor": mo.as_html(edge.numerator_expression()),
    }
    for edge in internal_edges(selected_diagram)
]

mo.vstack(
    [
        mo.md("### Interaction vertices"),
        mo.ui.table(
            _vertex_rows,
            column_widths={"vertex": 80, "interaction rule": 160},
        ),
        mo.md("### Internal propagators"),
        mo.ui.table(
            _propagator_rows,
            column_widths={"edge": 80, "loop field": 130},
        ),
    ],
    gap=1,
)

## Reading and reusing the result

For the gluon graph, `G` is the strong coupling, `f(...)` is the SU(3)
structure constant, and `Metric(...)` carries Lorentz contractions. The
ghost numerator exposes its momentum insertion, while the bottom loop also
carries Dirac and fundamental-color tensors. `Momentum(edge, index)` refers
to the momentum entering an instantiated rule through the indicated graph
edge. Sink/source indices make tensor contractions explicit.

The returned values are ordinary `symbolica.Expression` objects, so they
can be substituted, expanded, factored, differentiated, or passed directly
into the rest of a Symbolica calculation. Multiplying by
`selected_diagram.overall_factor_expression()` supplies the diagram-wide
combinatorial factor when constructing an integrand numerator.

In [ ]:
from symbolica import Expression, S
from symbolica.community.idenso import (
    simplify_color,
    simplify_gamma,
    simplify_metrics,
    to_dots,
)

_momentum = S("FeynKit::Momentum")
_sink_index = S("FeynKit::SinkIndex")
_metric = S("spenso::g")
_dot = S("spenso::dot")
_minkowski = S("spenso::mink")
_adjoint = S("spenso::coad")


def contract_qcd_numerator(expression: Expression) -> Expression:
    """Contract a FeynKit QCD bubble numerator with spenso/idenso."""
    contracted = simplify_metrics(expression.expand())
    contracted = simplify_gamma(contracted)
    contracted = simplify_color(contracted)
    return to_dots(simplify_metrics(contracted.expand()))


def project_gluon_self_energy_scalar(
    expression: Expression,
) -> Expression:
    """Apply the normalized transverse and color-singlet projector."""
    _mu = _sink_index(0, 1)
    _nu = _sink_index(1, 1)
    _color_a = _sink_index(0, 1)
    _color_b = _sink_index(1, 1)
    _external_momentum = _momentum(0, _minkowski(4))
    _momentum_squared = _dot(
        _external_momentum,
        _external_momentum,
    )
    _color_average = _metric(
        _adjoint(8, _color_a),
        _adjoint(8, _color_b),
    ) / 8
    _transverse_average = (
        _metric(_minkowski(4, _mu), _minkowski(4, _nu))
        - _momentum(0, _minkowski(4, _mu))
        * _momentum(0, _minkowski(4, _nu))
        / _momentum_squared
    ) / 3

    projected = simplify_metrics(
        (expression * _color_average * _transverse_average).expand()
    )
    projected = simplify_color(projected)
    return to_dots(simplify_metrics(projected.expand()))

## Project the contracted tensor to a diagnostic scalar

FeynKit lowers the model's UFO tensors while it instantiates each rule.
The numerator therefore already contains native four-dimensional
Minkowski and bispinor slots, SU(3) adjoint and fundamental slots, and the
color metrics that sew internal propagators. It also expands
$\not{p}=\gamma^\rho p_\rho$ with one private Lorentz index per
propagator. Stable `SourceIndex` and `SinkIndex` labels record which graph
endpoint owns every slot.

The actual algebra is then native idenso: metrics sew the propagator
endpoints, `simplify_gamma` closes the bottom-quark Dirac trace, and
`simplify_color` contracts the two structure constants or generators.
This first stage produces the tensor $N^{ab}_{\mu\nu}$ with only the two
external-gluon index pairs free.

For a scalar diagnostic at generic off-shell $p^2\ne0$, the next stage
applies the normalized 4D transverse, SU(3) color-singlet projector

$$
\mathcal P^{ab}_{\mu\nu}
  = \frac{\delta^{ab}}{8}\,
    \frac{1}{3}\left(g_{\mu\nu}
      - \frac{p_\mu p_\nu}{p^2}\right),
  \qquad p=\operatorname{Momentum}(0).
$$

Its normalization is
$1/[(d-1)(N_c^2-1)]=1/(3\cdot8)=1/24$. Native spenso metrics
identify and contract the external Lorentz and adjoint slots; idenso then
simplifies them and rewrites every momentum contraction as a symmetric
`spenso::dot`. The displayed result is therefore a scalar with no free
endpoint indices.

This is a projection of the selected **numerator**, not the unprojected
self-energy tensor or a claim that each loop class is separately
transverse. Although the gluon field is massless, the standard projector
above treats its self-energy momentum as off-shell; it is singular at
$p^2=0$. Individual contributions need not be transverse before the
appropriate gauge-sector sum and loop integration. Change `graph_index`
and rerun from the selection cell to project another loop contribution.

In [ ]:
contracted_numerator = contract_qcd_numerator(selected_numerator)
scalar_projection = project_gluon_self_energy_scalar(
    contracted_numerator
)

mo.ui.table(
    [
        {
            "contribution": selected_label,
            "normalized projector": mo.md(
                r"""\(\frac{\delta^{ab}}{8}\frac{1}{3}
                (g_{\mu\nu}-p_\mu p_\nu/p^2)\)"""
            ),
            "projected numerator": mo.as_html(scalar_projection),
            "result": "scalar; no free Lorentz or color indices",
        }
    ],
    column_widths={
        "contribution": 180,
        "normalized projector": 350,
        "projected numerator": 760,
        "result": 270,
    },
)